In [ ]:
## Python function using Apache Beam that performs a left join between two CSV files (stored in Google Cloud Storage) 
# based on a common key column and writes the joined data to a new CSV file.


In [58]:
import apache_beam as beam


def create_kv(line, column):
    try:
        key = line[column]
    except:
        key = ''
    return key, line

def join_rows(element, key_index_left, key_index_right, join_type='inner'):
    key, groups = element
    # print(key, groups)
    left_rows = groups.get('customers', [])
    right_rows = groups.get('transaction', [])
    results = []
    if join_type == 'inner':
        if not left_rows or not right_rows:
            return results
        for l in left_rows:
            for r in right_rows:
                if l[key_index_left] == r[key_index_right]:
                    r_trim = [v for i, v in enumerate(r) if i != key_index_right ]
                # else:
                #     r_trim = r
                results.append(l + r_trim)
    return results

pipeline = beam.Pipeline()

# Read the first CSV file
customers = (pipeline | "Read file 1" >> beam.io.ReadFromText(r'C:\Users\Prasad\Documents\python_tests\data\customers.csv', skip_header_lines=0)
                        | "Create columns" >> beam.Map(lambda x: x.split(','))
                        )

transaction = ( pipeline | "Read file 2" >> beam.io.ReadFromText(r'C:\Users\Prasad\Documents\python_tests\data\transactions.csv', skip_header_lines=0)
                        | "Create columns 2" >> beam.Map(lambda x: x.split(','))
                        )

kv1 = (
        customers | "Create KV 2" >> beam.Map(lambda line: create_kv(line, 0))
        )
kv2 = (
    transaction | "Create KV 1" >> beam.Map(lambda line: create_kv(line, 1))
)

joined = (
        {"customers": kv1, "transaction" : kv2}
        | "group data" >> beam.CoGroupByKey()
        | "join data" >> beam.FlatMap(lambda element: join_rows(element, 0, 1, 'inner') )
        # | "show joined" >> beam.Map(print)
        | "Create line" >> beam.Map(lambda line: ','.join(v for v in line))
        | 'WriteOutput' >> beam.io.WriteToText( r"C:\Users\Prasad\Documents\python_tests\data\cust_transactions",
                                                    file_name_suffix='.csv',
                                                    shard_name_template='')
)

pipeline.run()




In [51]:
import argparse
import io
import csv
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

# join_csvs_beam.py
#
# Usage example:
# python join_csvs_beam.py \
#   --input1 gs://my-bucket/path/to/files1-*.csv \
#   --input2 gs://my-bucket/path/to/files2-*.csv \
#   --key-index 0 \
#   --output gs://my-bucket/path/to/output/joined \
#   --runner DirectRunner
#
# For Dataflow, supply --runner DataflowRunner and Dataflow options.



def parse_csv_line(line, delimiter=','):
    # robust CSV parsing for a single line
    return next(csv.reader([line], delimiter=delimiter))


def to_kv(row, key_index):
    try:
        key = row[key_index]
    except IndexError:
        key = ''
    return key, row


def join_rows(element, key_index_left, key_index_right, join_type='inner'):
    key, groups = element
    left_rows = groups.get('left', [])
    right_rows = groups.get('right', [])

    results = []
    if join_type == 'inner':
        if not left_rows or not right_rows:
            return results
        for l in left_rows:
            for r in right_rows:
                # avoid duplicating key from right side if indices match
                if key_index_left == key_index_right:
                    r_trim = [v for i, v in enumerate(r) if i != key_index_right]
                else:
                    r_trim = r
                results.append(l + r_trim)
    elif join_type == 'left':
        if not left_rows:
            return results
        if not right_rows:
            for l in left_rows:
                results.append(l + ([''] * (len(right_rows[0]) if right_rows else 0)))
        else:
            for l in left_rows:
                for r in right_rows:
                    if key_index_left == key_index_right:
                        r_trim = [v for i, v in enumerate(r) if i != key_index_right]
                    else:
                        r_trim = r
                    results.append(l + r_trim)
    elif join_type == 'full':
        if left_rows and right_rows:
            for l in left_rows:
                for r in right_rows:
                    if key_index_left == key_index_right:
                        r_trim = [v for i, v in enumerate(r) if i != key_index_right]
                    else:
                        r_trim = r
                    results.append(l + r_trim)
        else:
            if left_rows:
                for l in left_rows:
                    results.append(l + ([''] * (len(right_rows[0]) if right_rows else 0)))
            if right_rows:
                for r in right_rows:
                    if key_index_left == key_index_right:
                        r_trim = [v for i, v in enumerate(r) if i != key_index_right]
                    else:
                        r_trim = r
                    results.append(([''] * (len(left_rows[0]) if left_rows else 0)) + r_trim)
    return results


def row_to_csv_line(row, delimiter=','):
    output = io.StringIO()
    writer = csv.writer(output, delimiter=delimiter, lineterminator='')
    writer.writerow(row)
    return output.getvalue()


def run(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument('--input1', default=r'C:\Users\Prasad\Documents\python_tests\data\customers.csv', help='GCS file pattern for first CSV (e.g. gs://bucket/path/*.csv)')
    parser.add_argument('--input2', default=r'C:\Users\Prasad\Documents\python_tests\data\transactions.csv', help='GCS file pattern for second CSV')
    parser.add_argument('--key-index', type=int, default=0, help='Zero-based key column index (assumed same for both)')
    parser.add_argument('--key-index2', type=int, default=1, help='Zero-based key column index (assumed same for both)')
    parser.add_argument('--output', default=r'C:\Users\Prasad\Documents\python_tests\data\cust_transactions', help='GCS output prefix (folder + filename prefix); .csv suffix will be added')
    parser.add_argument('--delimiter', default=',', help='CSV delimiter (default ,)')
    parser.add_argument('--runner', default='DirectRunner', help='Beam runner')
    parser.add_argument('--join-type', default='inner', choices=['inner', 'left', 'full'], help='Join type')
    parser.add_argument('--skip-header', action='store_true', help='Skip header row in inputs (assumes same header across files)')
    known_args, pipeline_args = parser.parse_known_args(argv)

    options = PipelineOptions(pipeline_args, runner=known_args.runner)

    with beam.Pipeline(options=options) as p:
        # Read input CSVs. skip_header_lines used if supported by your Beam version.
        read1 = p | 'ReadCSV1' >> beam.io.ReadFromText(known_args.input1, skip_header_lines=1 if known_args.skip_header else 0)
        read2 = p | 'ReadCSV2' >> beam.io.ReadFromText(known_args.input2, skip_header_lines=1 if known_args.skip_header else 0)

        parsed1 = read1 | 'ParseCSV1' >> beam.Map(lambda l: parse_csv_line(l, delimiter=known_args.delimiter))
        parsed2 = read2 | 'ParseCSV2' >> beam.Map(lambda l: parse_csv_line(l, delimiter=known_args.delimiter))

        kv1 = parsed1 | 'ToKV1' >> beam.Map(lambda row: to_kv(row, known_args.key_index))
        kv2 = parsed2 | 'ToKV2' >> beam.Map(lambda row: to_kv(row, known_args.key_index2))

        grouped = ({'left': kv1, 'right': kv2}
                   | 'CoGroup' >> beam.CoGroupByKey())

        joined = (grouped
                  | 'JoinRows' >> beam.FlatMap(
                        lambda elem, ki_left, ki_right, jt: join_rows(elem, ki_left, ki_right, jt),
                        ki_left=known_args.key_index,
                        ki_right=known_args.key_index2,
                        jt=known_args.join_type)
                    | 'show data' >> beam.Map(print)
                    )

        # csv_lines = joined | 'ToCSVLine' >> beam.Map(lambda row: row_to_csv_line(row, delimiter=known_args.delimiter))

        # # Optionally write a header if skip_header was True and you want to combine headers:
        # # For simplicity we don't auto-generate a header here.
        # csv_lines | 'WriteOutput' >> beam.io.WriteToText(
        #     known_args.output,
        #     file_name_suffix='.csv',
        #     shard_name_template='',
        # )


if __name__ == '__main__':
    run()

['customer_id', 'name', 'city', 'transaction_id', 'customer_id', 'amount', 'date']
['1', 'John Doe', 'Mumbai', '1001', '1', '540', '2024-01-12']
['1', 'John Doe', 'Mumbai', '1003', '1', '300', '2024-01-18']
['2', 'Aarti Sharma', 'Pune', '1002', '2', '1290', '2024-01-15']
['2', 'Aarti Sharma', 'Pune', '1008', '2', '450', '2024-01-22']
['3', 'Ravi Kumar', 'Delhi', '1005', '3', '1500', '2024-01-20']
['3', 'Ravi Kumar', 'Delhi', '1013', '3', '500', '2024-01-27']
['4', 'Sneha Patil', 'Nagpur', '1004', '4', '780', '2024-01-18']
['5', 'Michael Brown', 'Bangalore', '1007', '5', '900', '2024-01-22']
['6', 'Sara Wilson', 'Chennai', '1020', '6', '950', '2024-02-01']
['7', 'Amit Singh', 'Jaipur', '1006', '7', '220', '2024-01-21']
['8', 'Sunita Desai', 'Hyderabad', '1017', '8', '670', '2024-01-30']
['9', 'Vijay Nair', 'Kochi', '1009', '9', '180', '2024-01-24']
['11', 'Raj Malhotra', 'Indore', '1018', '11', '1200', '2024-01-31']
['12', 'Kevin Lewis', 'Kolkata', '1010', '12', '2400', '2024-01-25']
['

In [ ]:
import apache_beam as beam

pipeline = beam.Pipeline()
task1 = ( pipeline
		| "Read data" >> beam.io.ReadFromText(r"C:\Users\Prasad\Documents\python_tests\data\data.csv")
		| "Transform data" >> beam.Map(lambda x: x.upper())
		| "Write data" >> beam.io.WriteToText(r"C:\Users\Prasad\Documents\python_tests\data\data_upper.csv",
                                        shard_name_template='')
		
)

pipeline.run()